# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuan05-afk/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

Train honest models vs the Week-4 baseline on a **client-holdout** split.


## 1. Method choice and why

**Methods:** Logistic Regression (linear baseline), Decision Tree (readable rules),
Random Forest (non-linear interactions). Selection metric: **Precision@50**.

Why RF is the expected winner: interactions among impressions, freshness, and position are
nonlinear. Why we still train LR/DT: interpretability and a complexity check — if RF does not
clearly beat them and the rule baseline, complexity is not earned.


In [1]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

ROOT = Path('.').resolve()
# Support running from repo root or work/notebooks
if (ROOT / 'data/raw/content_refresh_anonymized.csv').exists():
    DATA = ROOT / 'data/raw/content_refresh_anonymized.csv'
elif (ROOT.parents[1] / 'data/raw/content_refresh_anonymized.csv').exists():
    ROOT = ROOT.parents[1]
    DATA = ROOT / 'data/raw/content_refresh_anonymized.csv'
else:
    # Colab / nested
    for cand in [ROOT, *ROOT.parents]:
        if (cand / 'data/raw/content_refresh_anonymized.csv').exists():
            ROOT = cand
            DATA = cand / 'data/raw/content_refresh_anonymized.csv'
            break
    else:
        raise FileNotFoundError('data/raw/content_refresh_anonymized.csv')

WORK_OUT = ROOT / 'work' / 'outputs'
WORK_OUT.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(DATA)
print(f'Loaded {len(df):,} rows × {df.shape[1]} cols from {DATA.relative_to(ROOT)}')
print(f'Clients (pseudonymous): {df["client_id"].nunique()}')

sys.path.insert(0, str(ROOT / 'scripts'))
# Prefer reference pipeline outputs if present
results_path = ROOT / 'outputs' / 'model_results.json'
if results_path.exists():
    results = json.loads(results_path.read_text())
    print('Loaded reference pipeline results from outputs/model_results.json')
    print('Best model:', results['best_model']['name'])
    print('Split:', results['split_strategy'])
else:
    results = None
    print('No model_results.json yet — will train inline.')


Loaded 30,000 rows × 44 cols from data\raw\content_refresh_anonymized.csv
Clients (pseudonymous): 32
Loaded reference pipeline results from outputs/model_results.json
Best model: random_forest
Split: client_holdout


## 2. Split design

**Strategy:** `client_holdout` — hold out entire clients for test so we measure
cross-client generalization, not page memorization within a client.

Train/test sizes and positive rates are reported from the reference run (seed-stable).


In [2]:
if results:
    print(f"train_rows={results['train_rows']} test_rows={results['test_rows']}")
    print(f"target_positive_rate={results['target_positive_rate']:.3f}")
    print(f"feature_count={results['feature_count']}")
    print('Held-out clients are unseen in training — reduces leakage from client-specific styles.')
else:
    print('Run: python scripts/run_all.py')


train_rows=27675 test_rows=2325
target_positive_rate=0.542
feature_count=52
Held-out clients are unseen in training — reduces leakage from client-specific styles.


## 3. Train + compare vs my baseline

Same split, same Precision@K. Numbers below are from the committed `outputs/model_results.json`
produced by `scripts/run_all.py` (reproducible).


In [3]:
rows = []
if results:
    b = results['baseline']
    rows.append({'model':'baseline_rules','precision_at_50':b['baseline_precision_at_50'],
                 'roc_auc':b['baseline_roc_auc'],'average_precision':b['baseline_average_precision']})
    for name, m in results['models'].items():
        rows.append({'model':name,'precision_at_50':m['precision_at_50'],
                     'roc_auc':m['roc_auc'],'average_precision':m['average_precision'],
                     'recall':m['recall'],'f1':m['f1']})
    comp = pd.DataFrame(rows).sort_values('precision_at_50', ascending=False)
    print(comp.to_string(index=False))
    print(f"\nBase rate: {results['target_positive_rate']:.3f}")
    print(f"Best by precision_at_50: {results['best_model']['name']}")
    (WORK_OUT / 'w05_model_comparison.json').write_text(json.dumps({
        'comparison': rows,
        'best': results['best_model']['name'],
        'base_rate': results['target_positive_rate'],
    }, indent=2))
else:
    print('Missing results — run pipeline first.')


              model  precision_at_50  roc_auc  average_precision   recall       f1
      random_forest             0.74 0.750030           0.618219 0.743674 0.639546
      decision_tree             0.62 0.741520           0.575319 0.716172 0.633885
logistic_regression             0.40 0.700291           0.521542 0.566557 0.566245
     baseline_rules             0.24 0.626892           0.467607      NaN      NaN

Base rate: 0.542
Best by precision_at_50: random_forest


## 4. Errors and interpretation

Top importances (RF): days_with_impressions, log_impressions_90d, avg_position, content_age_days.
Errors to expect: high-impression stable pages scored high (false positives) and low-volume
declines missed (false negatives). The playbook therefore requires human review on
high-confidence rows only.


In [4]:
if results:
    fi = results['best_model']['feature_importance_top'][:10]
    print('Top features:')
    for x in fi:
        print(f"  {x['feature']}: {x['importance']:.4f}")
    rf = results['models']['random_forest']
    base_p50 = results['baseline']['baseline_precision_at_50']
    print(f"\nLift Precision@50: {rf['precision_at_50']:.2f} vs baseline {base_p50:.2f} "
          f"(≈{rf['precision_at_50']/max(base_p50,1e-6):.1f}× on this holdout)")
    print('Honest read: strong lift on this starter holdout; still decision-support, not auto-publish.')


Top features:
  days_with_impressions: 0.1581
  log_impressions_90d: 0.1286
  avg_position: 0.1092
  content_age_days: 0.0952
  char_count: 0.0426
  word_count: 0.0396
  log_clicks_90d: 0.0345
  ctr: 0.0333
  scroll_rate: 0.0312
  days_with_sessions: 0.0280

Lift Precision@50: 0.74 vs baseline 0.24 (≈3.1× on this holdout)
Honest read: strong lift on this starter holdout; still decision-support, not auto-publish.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
